# Phase 2 Notebook: Candidate Generation Wikidata

## Principles
The goal is to find wikidata candidates for the mentions identified in the previous step. Example numbers:
* episodes: 2038
* publications: 2548
* seasons: 17
* persons: 10098
* topics: 10713

The goal is to be systematic and carefull, to put as little burden onto wikidata as possible. We should try to infer as much information from links as possible before we do any other searches, such as label searches or manual searches for the remaining items without candidates. 

## Approach (Simple Tree Expansion Workflow)
1. Load the broadcasting program seed(s) from `data/00_setup/broadcasting_programs.csv` and resolve their Wikidata Q-IDs.
2. For each seed Q-ID, check local cache first.
   - If cached entity data exists, use it.
   - If not cached, query Wikidata once and store the raw result as a new separate file.
3. Expand the Wikidata tree from each known Q-ID.
   - Expand outlinks (claims from the entity to other entities/properties).
   - Expand inlinks (entities that point to the current entity).
   - Persist each expansion query result as its own file before any aggregation.
4. After each expansion step, scan discovered labels/aliases/Q-IDs/properties for possible matches to our mention targets (episodes, publications, seasons, persons, topics).
5. If a candidate match is found, treat that matched Q-ID as a new expansion node and continue expansion (match-gated to conserve API calls).
6. Stop expanding when no new candidate-producing nodes are found or when configured depth/query limits are reached.
7. Maintain an aggregated index for fast lookup, but always keep per-query raw files as the source of truth.
   - If an aggregate file is corrupted, rebuild it from individual query result files without data loss.

### Cache And Storage Principles
- Cache-first: never hit Wikidata if sufficient local query results already exist.
- Append-only raw query storage: one file per query result.
- Rebuildable aggregates: derived files can be regenerated from raw query files at any time.
- Idempotent reruns: rerunning the workflow should reuse existing raw files and only fetch missing queries.

## 1) Project Setup
Resolve repository paths and make the source package importable in this notebook session.

In [13]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'data').exists() and (cur / 'speakermining' / 'src').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'speakermining' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

WindowsPath('C:/workspace/git/borgnetzwerk/speaker-mining')

## 2) Configure Workflow Parameters
Set rate limiting, cache behavior, and expansion limits before running discovery.
Note: `max_queries_per_run` is intentionally low while bugs are fixed and API usage stays minimal.

In [14]:
# Workflow Configuration
config = {
    "max_depth": 2,                          # Max tree expansion depth (0=seeds only, 1=neighbors, 2=2nd neighbors, etc.)
    "max_nodes": 500,                        # Max total nodes to expand
    "max_queries_per_run": 1,              # Max network queries (cached results don't count; 0=unlimited)
    "max_neighbors_per_match": 200,        # Cap neighbors enqueued per matched node (0=unlimited)
    "query_timeout_seconds": 30,             # Timeout per HTTP request
    "query_delay_seconds": 1.0,              # Delay between queries (rate limiting)
    "inlinks_limit": 200,                    # Max inlinks per entity (SPARQL LIMIT)
    "cache_max_age_days": 365,                # Cache age threshold (days); older records are refreshed
}

print("Workflow Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")


Workflow Configuration:
  max_depth: 2
  max_nodes: 500
  max_queries_per_run: 1
  max_neighbors_per_match: 200
  query_timeout_seconds: 30
  query_delay_seconds: 1.0
  inlinks_limit: 200
  cache_max_age_days: 365


## 3) Load Broadcasting Program Seeds
The seeds are the starting points for tree expansion.

In [15]:
from process.candidate_generation.wikidata.broadcasting_program import load_broadcasting_program_seeds
from process.candidate_generation.wikidata.common import canonical_qid

seeds = load_broadcasting_program_seeds(ROOT)
seed_qids = [canonical_qid(seed.get("wikidata_id", "")) for seed in seeds]
seed_qids = [qid for qid in seed_qids if qid]

print(f"Loaded {len(seeds)} broadcasting programs as seeds")
for seed in seeds:
    qid = canonical_qid(seed.get("wikidata_id", ""))
    if qid:
        print(f"\t• {seed.get('name', 'N/A')} ({qid})")


Loaded 1 broadcasting programs as seeds
	• Markus Lanz (Q1499182)


## 4) Load Mention Targets
These are the mentions from Phase 1 that we'll match against discovered Wikidata entities.

In [16]:
from process.candidate_generation.wikidata.episode import load_episode_targets, load_publication_targets
from process.candidate_generation.wikidata.person import load_person_targets
from process.candidate_generation.wikidata.season import load_season_targets
from process.candidate_generation.wikidata.topic import load_topic_targets

# Load all mention types
seasons = load_season_targets(ROOT)
episodes = load_episode_targets(ROOT)
publications = load_publication_targets(ROOT)
topics = load_topic_targets(ROOT)
persons = load_person_targets(ROOT)

mention_stats = {
    "Seasons": len(seasons),
    "Episodes": len(episodes),
    "Publications": len(publications),
    "Topics": len(topics),
    "Persons": len(persons),
}

print("\nMention Targets Loaded:")
print(f"  {'Type':<15} {'Count':<10}")
print("  " + "=" * 25)
for mention_type, count in mention_stats.items():
    print(f"  {mention_type:<15} {count:<10}")

total_mentions = sum(mention_stats.values())
print(f"  {'TOTAL':<15} {total_mentions:<10}")

# Show some examples from each type
print("\n\nExample Mentions:")

if seasons:
    print(f"\n  Seasons (first 3):")
    for s in seasons[:3]:
        print(f"    • {s['mention_label'][:50]}")

if episodes:
    print(f"\n  Episodes (first 3):")
    for ep in episodes[:3]:
        print(f"    • {ep['mention_label'][:50]}")

# Skip Publications since they are often not modeled in episodes
if publications:
    print(f"\n  Publications (first 3):")
    for p in publications[:3]:
        print(f"    • {p['mention_label'][:50]}")

if persons:
    print(f"\n  Persons (first 3):")
    for p in persons[:3]:
        print(f"    • {p['mention_label'][:50]}")

# Skip Topics for now until they are better chunked and filtered; they are very noisy
if topics:
    print(f"\n  Topics (first 3):")
    for t in topics[:3]:
        print(f"    • {t['mention_label'][:50]}")




Mention Targets Loaded:
  Type            Count     
  Seasons         17        
  Episodes        2038      
  Publications    2548      
  Topics          10713     
  Persons         10098     
  TOTAL           25414     


Example Mentions:

  Seasons (first 3):
    • Markus Lanz, Staffel 1
    • Markus Lanz, Staffel 2
    • Markus Lanz, Staffel 3

  Episodes (first 3):
    • Markus Lanz 03.06.2008
    • Markus Lanz 04.06.2008
    • Markus Lanz 05.06.2008

  Publications (first 3):
    • ZDF 03.06.2008 22:44:42
    • ZDF 04.06.2008 02:35:35
    • ZDFdokukanal 07.06.2008 17:03:31

  Persons (first 3):
    • Franjo Pooth
    • Georg PIEPER
    • Jelena WAHLER

  Topics (first 3):
    • Insolvenzaffäre von ihrem Ehemann Franjo Pooth
    • BORG (Hai-Schützer) und Folkart SCHWEIZER (wurde v
    • Gesundheitsmythen und Schönheitsoperationen


## 5) Execute Tree Expansion
Run the candidate generation workflow with the configured parameters above.

In [17]:
from process.candidate_generation.wikidata.bfs_expansion import run_bfs_expansion

print("Starting BFS tree expansion...\n")

# Combine all pre-loaded target rows into a single list
all_target_rows = episodes + publications + seasons + persons + topics

summary = run_bfs_expansion(
    ROOT,
    seeds=seeds,
    target_rows=all_target_rows,
    max_depth=config["max_depth"],
    max_nodes=config["max_nodes"],
    max_queries_per_run=config["max_queries_per_run"],
    max_neighbors_per_match=config["max_neighbors_per_match"],
    query_timeout_seconds=config["query_timeout_seconds"],
    query_delay_seconds=config["query_delay_seconds"],
    inlinks_limit=config["inlinks_limit"],
    cache_max_age_days=config["cache_max_age_days"],
)

print("\n\nExecution Summary:")
print("=" * 50)
for key, value in summary.items():
    if not key.endswith("_csv"):
        print(f"  {key:<30} {value}")

Starting BFS tree expansion...



Execution Summary:
  raw_files                      16
  candidate_rows                 8
  seed_qids                      ['Q1499182']
  expanded_nodes                 1
  network_queries                0
  discovered_candidates          0
  budget_exhausted               False
  queued_neighbors               0
  expanded_due_to_match          0
  skipped_duplicate_match_events 6
  new_unique_candidates          0
  max_depth                      2


## 6) Review Outputs
Load and inspect the generated candidates.

In [18]:
import pandas as pd
from pathlib import Path

# Load the generated candidates
candidates_path = ROOT / "data" / "20_candidate_generation" / "candidates.csv"
if candidates_path.exists():
    candidates_df = pd.read_csv(candidates_path)
    print(f"Loaded {len(candidates_df)} candidates from {candidates_path.name}\n")
    
    # Show statistics by mention type
    print("Candidates by Mention Type:")
    print(candidates_df["mention_type"].value_counts().to_string())
    
    print("\n\nSample Candidates (first 10):")
    display(candidates_df[["mention_id", "mention_type", "mention_label", "candidate_id", "candidate_label"]].head(10))
    
    # Statistics
    print(f"\n\nCandidate Coverage:")
    unique_mentions = candidates_df["mention_id"].nunique()
    total_mentions_in_target = total_mentions  # from cell 4
    coverage_pct = (unique_mentions / total_mentions_in_target * 100) if total_mentions_in_target > 0 else 0
    print(f"  Mentions with ≥1 candidate: {unique_mentions} / {total_mentions_in_target} ({coverage_pct:.1f}%)")
    print(f"  Avg candidates per mention: {len(candidates_df) / unique_mentions if unique_mentions > 0 else 0:.2f}")
else:
    print(f"⚠️  Candidates file not found at {candidates_path}")


Loaded 8 candidates from candidates.csv

Candidates by Mention Type:
mention_type
person     5
season     2
episode    1


Sample Candidates (first 10):


,mention_id,mention_type,mention_label,candidate_id,candidate_label
0,ep_77511f3fb329,episode,Markus Lanz,Q1499182,Markus Lanz
1,pm_58c8bb7d639d,person,Markus Lanz,Q1499182,Markus Lanz
2,pm_39ad5f27f679,person,Markus Lanz,Q1499182,Markus Lanz
3,pm_9afbf615be53,person,Markus Lanz,Q1499182,Markus Lanz
4,pm_8a856bb57dae,person,Markus Lanz,Q1499182,Markus Lanz
5,pm_a7a1d4545970,person,Markus Lanz,Q1499182,Markus Lanz
6,se_f6a1299d4bc2,season,"Markus Lanz, Staffel 17",Q130559283,"Markus Lanz, Staffel 17"
7,se_ad0039d2b7dc,season,"Markus Lanz, Staffel 16",Q132855776,"Markus Lanz, Staffel 16"




Candidate Coverage:
  Mentions with ≥1 candidate: 8 / 25414 (0.0%)
  Avg candidates per mention: 1.00


## ToDo
- Expand candidate matching beyond exact-normalized labels (fuzzy/diacritics).
- Review potential overmatching from signature set (Q-IDs, P-IDs, linked IDs).
- Revisit aggregate deduplication to preserve provenance for multiple sources/contexts.
- Rank/prioritize neighbors before enqueueing to preserve budget for likely matches.
- Validate inlink payloads before use (guard against malformed or partial bindings).